# 1. What Is an API?

An **API** is an Application Programming Interface. It is a way for one program to ask another program for data or ask it to do work.

When we use a web API, Python acts like a client:

- Python sends a **request** to a URL.
- The API server sends back a **response**.
- Many APIs send data back as **JSON**, which looks a lot like Python dictionaries and lists.


## Today's Goals

- Make a request to a real public API.
- Read the response status code.
- Check a failed request before reading JSON.
- Turn JSON into Python data.
- Pull one useful value out of a response.


In [1]:
import requests
from pprint import pprint

TIMEOUT = 10

## Our First API Request

This API looks up information for a ZIP code. It does not need an account, password, or API key.


In [17]:
url = "https://api.zippopotam.us/us/90403"
response = requests.get(url, timeout=TIMEOUT)

In [18]:
response.url

'https://api.zippopotam.us/us/90403'

In [19]:
print(response.status_code)
print(response.url)

200
https://api.zippopotam.us/us/90403


A status code of `200` means the request worked.

The response body is text, even when it looks like structured data.


In [20]:
response.text

'{"country": "United States", "country abbreviation": "US", "post code": "90403", "places": [{"place name": "Santa Monica", "longitude": "-118.4924", "latitude": "34.0287", "state": "California", "state abbreviation": "CA"}]}'

In [21]:
print(type(response.text))
print(response.text[:20])

<class 'str'>
{"country": "United 


In [24]:
response.json()

{'country': 'United States',
 'country abbreviation': 'US',
 'post code': '90403',
 'places': [{'place name': 'Santa Monica',
   'longitude': '-118.4924',
   'latitude': '34.0287',
   'state': 'California',
   'state abbreviation': 'CA'}]}

## When a Request Does Not Work

`raise_for_status()` tells Python to raise an error for responses like `404` or `500`.

Here we ask the same ZIP-code API for a ZIP code that does not exist, then catch the error so the notebook keeps running.


In [ ]:
bad_url = "https://api.zippopotam.us/us/0"
response = requests.get(bad_url, timeout=TIMEOUT)

try:
    response.raise_for_status()
    print("Request worked")
except requests.HTTPError as error:
    print("Request did not work")
    print(error)


Request did not work
404 Client Error: Not Found for url: https://api.zippopotam.us/us/0


## JSON: API Data Python Can Use

`response.json()` converts the JSON response into Python dictionaries and lists.


In [28]:
url = "https://api.zippopotam.us/us/90403"
response = requests.get(url, timeout=TIMEOUT)

In [29]:
data = response.json()

print(type(data))
pprint(data)

<class 'dict'>
{'country': 'United States',
 'country abbreviation': 'US',
 'places': [{'latitude': '34.0287',
             'longitude': '-118.4924',
             'place name': 'Santa Monica',
             'state': 'California',
             'state abbreviation': 'CA'}],
 'post code': '90403'}


Now we can use normal Python indexing to get specific details.


In [35]:
data

{'country': 'United States',
 'country abbreviation': 'US',
 'post code': '90403',
 'places': [{'place name': 'Santa Monica',
   'longitude': '-118.4924',
   'latitude': '34.0287',
   'state': 'California',
   'state abbreviation': 'CA'}]}

In [36]:
place = data["places"][0]

city = place["place name"]
state = place["state"]
country = data["country"]

print(f"ZIP code {data['post code']} is in {city}, {state}, {country}.")

ZIP code 90403 is in Santa Monica, California, United States.


## Another Public API

This API returns public holidays for a country and year. The URL itself contains the country code and year.


In [39]:
holidays_url = "https://date.nager.at/api/v3/PublicHolidays/2026/US"
holidays_response = requests.get(holidays_url, timeout=TIMEOUT)

print(holidays_response.status_code)

200


In [40]:
holidays = holidays_response.json()

holidays

[{'date': '2026-01-01',
  'localName': "New Year's Day",
  'name': "New Year's Day",
  'countryCode': 'US',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Public', 'Bank']},
 {'date': '2026-01-19',
  'localName': 'Martin Luther King, Jr. Day',
  'name': 'Martin Luther King, Jr. Day',
  'countryCode': 'US',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Public', 'Bank']},
 {'date': '2026-02-12',
  'localName': "Lincoln's Birthday",
  'name': "Lincoln's Birthday",
  'countryCode': 'US',
  'fixed': False,
  'global': False,
  'counties': ['US-CA',
   'US-CT',
   'US-IL',
   'US-IN',
   'US-KY',
   'US-MI',
   'US-NY',
   'US-MO',
   'US-OH'],
  'launchYear': None,
  'types': ['Observance']},
 {'date': '2026-02-16',
  'localName': "Washington's Birthday",
  'name': 'Presidents Day',
  'countryCode': 'US',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Public',

In [48]:
for holiday in holidays:
    if holiday["date"][5:7] == "06":
        print(holiday)

{'date': '2026-06-19', 'localName': 'Juneteenth National Independence Day', 'name': 'Juneteenth National Independence Day', 'countryCode': 'US', 'fixed': False, 'global': True, 'counties': None, 'launchYear': None, 'types': ['Public', 'Bank']}


## Vocabulary Check

- **Client**: the program making the request. In this class, that is Python.
- **Server**: the computer that receives the request and sends a response.
- **Request**: the message Python sends to an API.
- **Response**: the message the API sends back.
- **JSON**: a common data format for API responses.
- **Status code**: a number that tells us whether the request worked.
